# 第 27 章 · 显存：谁占了、怎么省、怎么看

本章回答：训练时显存都被谁占了？激活为什么常常比参数还大？激活重计算（activation checkpointing）用多少算力换多少显存？OOM 了怎么看清是谁、按什么顺序去省？

配套讲义：仓库里 `pytorch-deep-dive/notes/lessons/27-memory.md`（讲解、动画、练习都在讲义里）。对应原文：Ultra-Scale Playbook §Memory usage in Transformers（`ultrascale-playbook.md` 第 208 行起：Profiling the memory usage 第 242 行、Weights/grads/optimizer 第 270 行、Activations memory 第 350 行、Activation recomputation 第 387 行、Gradient accumulation 第 466 行）；AISPE ch13「PyTorch 内存分析和调优」（`ch13.md` 第 1377–1545 行）；多机报告 §6 显存优化清单；本章激活字节数是本机 torch 2.15.0 用 `saved_tensors_hooks` 实测

**怎么用**：在 `pytorch-deep-dive/solutions/chapter-code/` 下用 `uv run jupyter lab` 打开（或任何装了 torch 的环境）。从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。Mac 上能跑：计时类的格子在 Mac 上用 MPS，分布式的格子在本机起 2 个 Gloo 进程。

In [ ]:
from torch.utils.checkpoint import checkpoint
import torch
import torch.nn as nn

torch.set_printoptions(precision=4, sci_mode=False)
print("准备好了", torch.__version__)

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 公式 / 说法 | 写法 | 跑出来 |
|---|---|---|---|
| 第一步 激活估算 | s·b·h·34 | `layer_activation_bytes(4096, 1, 4096)` | 每层 570 MB，32 层 18.3 GB |
| 第二步 重计算 | 只存块输入 | `saved_activation_mb(False / True)` | 9.125 MB → 1.125 MB |

## 1. 先把要用的东西定义好

这几格只是定义（函数、类、常量），运行后没有输出也正常。用到的前几章的函数也一起抄进来了，不需要本地文件。

**函数 `layer_activation_bytes`**（来自 `ch27_memory.py`）：每层激活字节估算 s·b·h·(34 + 5as/h)；a = 0 表示只算第一项（用了 FlashAttention）。

In [ ]:
def layer_activation_bytes(s: int, b: int, h: int, a: int = 0) -> float:
    """每层激活字节估算 s·b·h·(34 + 5as/h)；a = 0 表示只算第一项（用了 FlashAttention）。"""
    return s * b * h * (34 + 5 * a * s / h)

**函数 `saved_activation_mb`**（来自 `ch27_memory.py`）：统计为反向存下的张量字节（不算参数本身）。

In [ ]:
def saved_activation_mb(use_checkpoint: bool, B=32, D=1024, L=8) -> float:
    """统计为反向存下的张量字节（不算参数本身）。"""
    torch.manual_seed(0)
    blocks = nn.ModuleList([nn.Sequential(nn.Linear(D, 4 * D), nn.GELU(), nn.Linear(4 * D, D)) for _ in range(L)])
    params = {p.untyped_storage().data_ptr() for p in blocks.parameters()}
    x = torch.randn(B, D, requires_grad=True)
    saved = [0]

    def pack(t):
        if t.untyped_storage().data_ptr() not in params:
            saved[0] += t.numel() * t.element_size()
        return t

    with torch.autograd.graph.saved_tensors_hooks(pack, lambda t: t):
        h = x
        for blk in blocks:
            h = checkpoint(blk, h, use_reentrant=False) if use_checkpoint else blk(h)
        loss = h.pow(2).mean()
    loss.backward()
    return saved[0] / 2**20

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
per = layer_activation_bytes(4096, 1, 4096)

In [ ]:
print("\n第一步 7B（h=4096）、s=4096、b=1：每层", round(per / 1e6), "MB，32 层", round(per * 32 / 1e9, 1), "GB")

In [ ]:
print("\n第二步 存下的激活：不重计算", saved_activation_mb(False), "MB，checkpoint", saved_activation_mb(True), "MB")

## 3. 代码里才看得到的

1. **统计时去掉了参数本身**：Linear 的反向也要用权重，但权重本来就在显存里，不算激活。
2. **重算时存下的张量 hook 看不到**：它们在反向里临时出现、用完就丢，所以 checkpoint 只统计到输入。

## 4. 自己改一改

- `saved_activation_mb(False, L=16)`，看是不是线性增长。
- 只对偶数层用 checkpoint，看结果在两者之间。

讲义末尾的练习题可以在下面的空格子里用上面的函数验算——先自己在纸上算，再来核对。

In [ ]:
# 在这里写你的代码

## 口诀

> 显存四块：模型状态十六 Ψ、激活随 b s h L、临时和碎片；长序列激活比参数大；
> 重计算只存每块输入，反向再跑一遍前向，多三分之一算力；
> OOM 先录快照看是谁；bf16、Flash、减 batch、重计算、FSDP，offload 最后。